# Comprehensive Anomaly Detection System for Time Series

This notebook implements a multi-method anomaly detection system with:
- Statistical methods (Z-score, IQR, Grubbs test)
- Machine learning methods (Isolation Forest, LOF, One-Class SVM)
- Time series specific methods (S-H-ESD, Matrix Profile, LSTM Autoencoder)
- Ensemble methods combining multiple detectors
- Real-time anomaly detection
- Evaluation and visualization tools

In [ ]:
# Import required libraries
import warnings
from dataclasses import dataclass, field

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.signal import find_peaks

warnings.filterwarnings("ignore")

# Machine learning imports

# Visualization
import plotly.graph_objects as go
import stumpy  # For Matrix Profile

# Deep learning
from plotly.subplots import make_subplots
from sklearn.covariance import EllipticEnvelope
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

# Time series specific
from statsmodels.tsa.seasonal import STL
from tensorflow import keras
from tensorflow.keras import layers

# Set style
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

## 1. Core Anomaly Detection Classes

In [ ]:
@dataclass
class AnomalyResult:
    """Container for anomaly detection results."""

    method: str
    anomalies: np.ndarray
    scores: np.ndarray | None = None
    threshold: float | None = None
    metadata: dict = field(default_factory=dict)

    @property
    def anomaly_rate(self) -> float:
        """Calculate the percentage of anomalies."""
        return np.mean(self.anomalies) * 100

    @property
    def anomaly_indices(self) -> np.ndarray:
        """Get indices of anomalies."""
        return np.where(self.anomalies)[0]


class AnomalyDetector:
    """Multi-method anomaly detection system for time series."""

    def __init__(self, contamination: float = 0.1, sensitivity: float = 0.95):
        """Initialize the anomaly detector.

        Parameters:
        -----------
        contamination : float
            Expected proportion of anomalies in the dataset
        sensitivity : float
            Sensitivity level for detection (0-1)
        """
        self.contamination = contamination
        self.sensitivity = sensitivity
        self.results = {}
        self.fitted_models = {}

    # ========== Statistical Methods ==========

    def zscore_detection(
        self, data: pd.Series, threshold: float = 3.0
    ) -> AnomalyResult:
        """Z-score based anomaly detection.

        Parameters:
        -----------
        data : pd.Series
            Time series data
        threshold : float
            Z-score threshold for anomaly detection
        """
        values = data.values
        mean = np.mean(values)
        std = np.std(values)

        z_scores = np.abs((values - mean) / std)
        anomalies = z_scores > threshold

        return AnomalyResult(
            method="Z-Score",
            anomalies=anomalies,
            scores=z_scores,
            threshold=threshold,
            metadata={"mean": mean, "std": std},
        )

    def iqr_detection(self, data: pd.Series, k: float = 1.5) -> AnomalyResult:
        """Interquartile Range (IQR) based anomaly detection.

        Parameters:
        -----------
        data : pd.Series
            Time series data
        k : float
            IQR multiplier for determining outliers
        """
        values = data.values
        Q1 = np.percentile(values, 25)
        Q3 = np.percentile(values, 75)
        IQR = Q3 - Q1

        lower_bound = Q1 - k * IQR
        upper_bound = Q3 + k * IQR

        anomalies = (values < lower_bound) | (values > upper_bound)
        scores = np.maximum(lower_bound - values, values - upper_bound)
        scores[~anomalies] = 0

        return AnomalyResult(
            method="IQR",
            anomalies=anomalies,
            scores=scores,
            threshold=k,
            metadata={
                "Q1": Q1,
                "Q3": Q3,
                "IQR": IQR,
                "lower_bound": lower_bound,
                "upper_bound": upper_bound,
            },
        )

    def grubbs_test(self, data: pd.Series, alpha: float = 0.05) -> AnomalyResult:
        """Grubbs test for outliers.

        Parameters:
        -----------
        data : pd.Series
            Time series data
        alpha : float
            Significance level
        """
        values = data.values.copy()
        anomalies = np.zeros(len(values), dtype=bool)

        outlier_indices = []
        while True:
            n = len(values[~anomalies])
            if n < 3:
                break

            mean = np.mean(values[~anomalies])
            std = np.std(values[~anomalies])

            # Calculate Grubbs statistic
            abs_dev = np.abs(values - mean)
            abs_dev[anomalies] = 0
            max_idx = np.argmax(abs_dev)
            G = abs_dev[max_idx] / std

            # Critical value
            t_dist = stats.t.ppf(1 - alpha / (2 * n), n - 2)
            G_critical = ((n - 1) / np.sqrt(n)) * np.sqrt(
                t_dist**2 / (n - 2 + t_dist**2)
            )

            if G > G_critical:
                anomalies[max_idx] = True
                outlier_indices.append(max_idx)
            else:
                break

        return AnomalyResult(
            method="Grubbs Test",
            anomalies=anomalies,
            scores=np.abs(values - np.mean(values)) / np.std(values),
            threshold=alpha,
            metadata={"outlier_indices": outlier_indices, "alpha": alpha},
        )

    # ========== Machine Learning Methods ==========

    def isolation_forest(
        self, data: pd.Series, n_estimators: int = 100
    ) -> AnomalyResult:
        """Isolation Forest anomaly detection.

        Parameters:
        -----------
        data : pd.Series
            Time series data
        n_estimators : int
            Number of trees in the forest
        """
        values = data.values.reshape(-1, 1)

        model = IsolationForest(
            contamination=self.contamination, n_estimators=n_estimators, random_state=42
        )

        predictions = model.fit_predict(values)
        anomalies = predictions == -1
        scores = -model.score_samples(values)  # Negative for consistency

        self.fitted_models["isolation_forest"] = model

        return AnomalyResult(
            method="Isolation Forest",
            anomalies=anomalies,
            scores=scores,
            metadata={"n_estimators": n_estimators},
        )

    def local_outlier_factor(
        self, data: pd.Series, n_neighbors: int = 20
    ) -> AnomalyResult:
        """Local Outlier Factor (LOF) anomaly detection.

        Parameters:
        -----------
        data : pd.Series
            Time series data
        n_neighbors : int
            Number of neighbors to consider
        """
        values = data.values.reshape(-1, 1)

        model = LocalOutlierFactor(
            n_neighbors=n_neighbors, contamination=self.contamination, novelty=False
        )

        predictions = model.fit_predict(values)
        anomalies = predictions == -1
        scores = -model.negative_outlier_factor_

        return AnomalyResult(
            method="Local Outlier Factor",
            anomalies=anomalies,
            scores=scores,
            metadata={"n_neighbors": n_neighbors},
        )

    def elliptic_envelope(self, data: pd.Series) -> AnomalyResult:
        """Elliptic Envelope (Minimum Covariance Determinant) anomaly detection.
        """
        values = data.values.reshape(-1, 1)

        model = EllipticEnvelope(contamination=self.contamination, random_state=42)

        predictions = model.fit_predict(values)
        anomalies = predictions == -1
        scores = -model.score_samples(values)

        self.fitted_models["elliptic_envelope"] = model

        return AnomalyResult(
            method="Elliptic Envelope", anomalies=anomalies, scores=scores
        )

    def one_class_svm(self, data: pd.Series, kernel: str = "rbf") -> AnomalyResult:
        """One-Class SVM anomaly detection.

        Parameters:
        -----------
        data : pd.Series
            Time series data
        kernel : str
            Kernel type ('rbf', 'linear', 'poly', 'sigmoid')
        """
        values = data.values.reshape(-1, 1)

        # Scale the data
        scaler = StandardScaler()
        values_scaled = scaler.fit_transform(values)

        model = OneClassSVM(kernel=kernel, nu=self.contamination, gamma="auto")

        predictions = model.fit_predict(values_scaled)
        anomalies = predictions == -1
        scores = -model.score_samples(values_scaled)

        self.fitted_models["one_class_svm"] = (model, scaler)

        return AnomalyResult(
            method="One-Class SVM",
            anomalies=anomalies,
            scores=scores,
            metadata={"kernel": kernel},
        )

    # ========== Time Series Specific Methods ==========

    def seasonal_hybrid_esd(
        self, data: pd.Series, seasonal_period: int = None
    ) -> AnomalyResult:
        """Seasonal Hybrid ESD (S-H-ESD) algorithm for seasonal time series.

        Parameters:
        -----------
        data : pd.Series
            Time series data
        seasonal_period : int
            Seasonal period (if None, will be auto-detected)
        """
        if seasonal_period is None:
            # Auto-detect seasonal period

            seasonal_period = self._detect_seasonality(data)

        # Perform STL decomposition
        stl = STL(data, seasonal=seasonal_period if seasonal_period else 13)
        result = stl.fit()

        # Get residuals
        residuals = result.resid

        # Apply Generalized ESD test on residuals
        max_outliers = int(len(data) * self.contamination)
        anomalies = self._generalized_esd(residuals.values, max_outliers)

        # Calculate anomaly scores
        scores = np.abs(residuals.values)

        return AnomalyResult(
            method="Seasonal Hybrid ESD",
            anomalies=anomalies,
            scores=scores,
            metadata={
                "seasonal_period": seasonal_period,
                "trend": result.trend,
                "seasonal": result.seasonal,
                "residual": result.resid,
            },
        )

    def _generalized_esd(self, data: np.ndarray, max_outliers: int) -> np.ndarray:
        """Generalized ESD test for outliers."""
        n = len(data)
        outliers = np.zeros(n, dtype=bool)

        for i in range(max_outliers):
            if np.sum(~outliers) < 3:
                break

            mean = np.mean(data[~outliers])
            std = np.std(data[~outliers])

            if std == 0:
                break

            # Calculate test statistic
            abs_dev = np.abs(data - mean)
            abs_dev[outliers] = 0  # Ignore already detected outliers

            max_idx = np.argmax(abs_dev)
            max_dev = abs_dev[max_idx]

            # Critical value (using t-distribution)
            alpha = 0.05
            p = 1 - alpha / (2 * (n - i))
            t_critical = stats.t.ppf(p, n - i - 2)

            # Lambda critical value
            lambda_critical = (
                (n - i - 1)
                * t_critical
                / np.sqrt((n - i) * ((n - i - 2) + t_critical**2))
            )

            # Test statistic
            test_stat = max_dev / std

            if test_stat > lambda_critical:
                outliers[max_idx] = True
            else:
                break

        return outliers

    def matrix_profile_anomalies(
        self, data: pd.Series, window_size: int = None
    ) -> AnomalyResult:
        """Matrix Profile based anomaly detection.

        Parameters:
        -----------
        data : pd.Series
            Time series data
        window_size : int
            Subsequence window size (if None, auto-determined)
        """
        values = data.values

        # Determine window size
        if window_size is None:
            window_size = max(4, min(len(values) // 4, 100))

        # Calculate matrix profile
        mp = stumpy.stump(values, m=window_size)

        # Matrix profile distances
        mp_dist = mp[:, 0]

        # Identify anomalies based on matrix profile values
        threshold = np.percentile(mp_dist, 100 * (1 - self.contamination))

        # Create full-length anomaly array
        anomalies = np.zeros(len(values), dtype=bool)

        # Mark subsequences as anomalous
        for i in range(len(mp_dist)):
            if mp_dist[i] > threshold:
                anomalies[i : i + window_size] = True

        return AnomalyResult(
            method="Matrix Profile",
            anomalies=anomalies,
            scores=np.concatenate([mp_dist, np.zeros(len(values) - len(mp_dist))]),
            threshold=threshold,
            metadata={"window_size": window_size, "matrix_profile": mp},
        )

    def _detect_seasonality(self, data: pd.Series) -> int:
        """Auto-detect seasonal period."""
        from statsmodels.tsa.stattools import acf

        # Calculate ACF
        acf_values = acf(data.dropna(), nlags=min(len(data) // 2, 100))

        # Find peaks in ACF
        peaks, _ = find_peaks(acf_values[1:], height=0.3)

        if len(peaks) > 0:
            return peaks[0] + 1
        return None

    # ========== Deep Learning Methods ==========

    def lstm_autoencoder_anomalies(
        self, data: pd.Series, sequence_length: int = 30, epochs: int = 50
    ) -> AnomalyResult:
        """LSTM Autoencoder for anomaly detection.

        Parameters:
        -----------
        data : pd.Series
            Time series data
        sequence_length : int
            Length of sequences for LSTM
        epochs : int
            Number of training epochs
        """
        values = data.values

        # Scale data
        scaler = StandardScaler()
        scaled_values = scaler.fit_transform(values.reshape(-1, 1)).flatten()

        # Create sequences
        sequences = self._create_sequences(scaled_values, sequence_length)

        # Build LSTM Autoencoder
        model = self._build_lstm_autoencoder(sequence_length)

        # Train model
        history = model.fit(
            sequences,
            sequences,
            epochs=epochs,
            batch_size=32,
            validation_split=0.1,
            verbose=0,
        )

        # Get reconstruction error
        predictions = model.predict(sequences, verbose=0)
        mse = np.mean(np.power(sequences - predictions, 2), axis=1)

        # Determine threshold
        threshold = np.percentile(mse, 100 * (1 - self.contamination))

        # Create full-length anomaly array
        anomalies = np.zeros(len(values), dtype=bool)
        anomalies[sequence_length:] = mse > threshold

        # Create full-length scores
        scores = np.zeros(len(values))
        scores[sequence_length:] = mse

        self.fitted_models["lstm_autoencoder"] = (model, scaler)

        return AnomalyResult(
            method="LSTM Autoencoder",
            anomalies=anomalies,
            scores=scores,
            threshold=threshold,
            metadata={
                "sequence_length": sequence_length,
                "epochs": epochs,
                "history": history.history,
            },
        )

    def _create_sequences(self, data: np.ndarray, sequence_length: int) -> np.ndarray:
        """Create sequences for LSTM input."""
        sequences = []
        for i in range(len(data) - sequence_length):
            sequences.append(data[i : i + sequence_length])
        return np.array(sequences)

    def _build_lstm_autoencoder(self, sequence_length: int) -> keras.Model:
        """Build LSTM Autoencoder model."""
        model = keras.Sequential(
            [
                # Encoder
                layers.LSTM(
                    32,
                    activation="relu",
                    input_shape=(sequence_length, 1),
                    return_sequences=True,
                ),
                layers.LSTM(16, activation="relu", return_sequences=False),
                # Decoder
                layers.RepeatVector(sequence_length),
                layers.LSTM(16, activation="relu", return_sequences=True),
                layers.LSTM(32, activation="relu", return_sequences=True),
                layers.TimeDistributed(layers.Dense(1)),
            ]
        )

        model.compile(optimizer="adam", loss="mse")
        return model

    # ========== Ensemble Methods ==========

    def ensemble_detection(
        self, data: pd.Series, methods: list[str] = None, voting: str = "majority"
    ) -> AnomalyResult:
        """Ensemble anomaly detection combining multiple methods.

        Parameters:
        -----------
        data : pd.Series
            Time series data
        methods : List[str]
            List of methods to use (if None, uses all available)
        voting : str
            Voting method ('majority', 'unanimous', 'weighted')
        """
        if methods is None:
            methods = [
                "zscore",
                "iqr",
                "isolation_forest",
                "local_outlier_factor",
                "matrix_profile",
            ]

        # Run selected methods
        method_results = {}
        for method in methods:
            if method == "zscore":
                method_results[method] = self.zscore_detection(data)
            elif method == "iqr":
                method_results[method] = self.iqr_detection(data)
            elif method == "isolation_forest":
                method_results[method] = self.isolation_forest(data)
            elif method == "local_outlier_factor":
                method_results[method] = self.local_outlier_factor(data)
            elif method == "matrix_profile":
                method_results[method] = self.matrix_profile_anomalies(data)

        # Combine results based on voting method
        anomaly_matrix = np.array(
            [result.anomalies for result in method_results.values()]
        )

        if voting == "majority":
            # Majority voting
            anomalies = np.sum(anomaly_matrix, axis=0) > len(methods) / 2
        elif voting == "unanimous":
            # All methods must agree
            anomalies = np.all(anomaly_matrix, axis=0)
        elif voting == "weighted":
            # Weighted voting based on method performance
            weights = self._calculate_method_weights(method_results)
            weighted_votes = np.average(
                anomaly_matrix.astype(float), axis=0, weights=weights
            )
            anomalies = weighted_votes > 0.5
        else:
            raise ValueError(f"Unknown voting method: {voting}")

        # Calculate ensemble scores
        scores = np.mean(
            [
                result.scores
                for result in method_results.values()
                if result.scores is not None
            ],
            axis=0,
        )

        return AnomalyResult(
            method=f"Ensemble ({voting})",
            anomalies=anomalies,
            scores=scores,
            metadata={
                "methods": methods,
                "voting": voting,
                "individual_results": method_results,
            },
        )

    def _calculate_method_weights(self, method_results: dict) -> np.ndarray:
        """Calculate weights for weighted voting based on method consistency."""
        weights = []
        for method, result in method_results.items():
            # Weight based on contamination rate closeness to expected
            rate_diff = abs(result.anomaly_rate / 100 - self.contamination)
            weight = 1 / (1 + rate_diff)
            weights.append(weight)

        # Normalize weights
        weights = np.array(weights)
        return weights / np.sum(weights)

    # ========== Comprehensive Detection ==========

    def detect_all(
        self, data: pd.Series, include_deep_learning: bool = False
    ) -> pd.DataFrame:
        """Run all anomaly detection methods and create comprehensive report.

        Parameters:
        -----------
        data : pd.Series
            Time series data
        include_deep_learning : bool
            Whether to include deep learning methods (slower)
        """
        print("Running comprehensive anomaly detection...\n")

        # Statistical methods
        print("Statistical methods:")
        self.results["zscore"] = self.zscore_detection(data)
        print(f"  ✓ Z-Score: {self.results['zscore'].anomaly_rate:.1f}% anomalies")

        self.results["iqr"] = self.iqr_detection(data)
        print(f"  ✓ IQR: {self.results['iqr'].anomaly_rate:.1f}% anomalies")

        self.results["grubbs"] = self.grubbs_test(data)
        print(f"  ✓ Grubbs: {self.results['grubbs'].anomaly_rate:.1f}% anomalies")

        # Machine learning methods
        print("\nMachine Learning methods:")
        self.results["isolation_forest"] = self.isolation_forest(data)
        print(
            f"  ✓ Isolation Forest: {self.results['isolation_forest'].anomaly_rate:.1f}% anomalies"
        )

        self.results["lof"] = self.local_outlier_factor(data)
        print(f"  ✓ LOF: {self.results['lof'].anomaly_rate:.1f}% anomalies")

        self.results["elliptic"] = self.elliptic_envelope(data)
        print(
            f"  ✓ Elliptic Envelope: {self.results['elliptic'].anomaly_rate:.1f}% anomalies"
        )

        self.results["svm"] = self.one_class_svm(data)
        print(f"  ✓ One-Class SVM: {self.results['svm'].anomaly_rate:.1f}% anomalies")

        # Time series specific
        print("\nTime Series specific methods:")
        try:
            self.results["seasonal_esd"] = self.seasonal_hybrid_esd(data)
            print(
                f"  ✓ Seasonal ESD: {self.results['seasonal_esd'].anomaly_rate:.1f}% anomalies"
            )
        except Exception as e:
            print(f"  ✗ Seasonal ESD failed: {str(e)[:50]}")

        try:
            self.results["matrix_profile"] = self.matrix_profile_anomalies(data)
            print(
                f"  ✓ Matrix Profile: {self.results['matrix_profile'].anomaly_rate:.1f}% anomalies"
            )
        except Exception as e:
            print(f"  ✗ Matrix Profile failed: {str(e)[:50]}")

        # Deep learning (optional)
        if include_deep_learning:
            print("\nDeep Learning methods:")
            try:
                self.results["lstm_autoencoder"] = self.lstm_autoencoder_anomalies(data)
                print(
                    f"  ✓ LSTM Autoencoder: {self.results['lstm_autoencoder'].anomaly_rate:.1f}% anomalies"
                )
            except Exception as e:
                print(f"  ✗ LSTM Autoencoder failed: {str(e)[:50]}")

        # Ensemble
        print("\nEnsemble methods:")
        self.results["ensemble"] = self.ensemble_detection(data)
        print(f"  ✓ Ensemble: {self.results['ensemble'].anomaly_rate:.1f}% anomalies")

        # Create report DataFrame
        report = self._create_anomaly_report(data)

        print("\nDetection complete!")
        return report

    def _create_anomaly_report(self, data: pd.Series) -> pd.DataFrame:
        """Create comprehensive anomaly report DataFrame."""
        report = pd.DataFrame(index=data.index)
        report["value"] = data.values

        # Add results from each method
        for method_name, result in self.results.items():
            report[f"{method_name}_anomaly"] = result.anomalies
            if result.scores is not None:
                report[f"{method_name}_score"] = result.scores

        # Add summary columns
        anomaly_cols = [col for col in report.columns if col.endswith("_anomaly")]
        report["anomaly_count"] = report[anomaly_cols].sum(axis=1)
        report["anomaly_percentage"] = (
            report["anomaly_count"] / len(anomaly_cols)
        ) * 100
        report["is_anomaly"] = report["anomaly_percentage"] > 50  # Majority voting

        return report

## 2. Visualization Tools

In [ ]:
class AnomalyVisualizer:
    """Visualization tools for anomaly detection results."""

    @staticmethod
    def plot_anomalies(
        data: pd.Series,
        result: AnomalyResult,
        title: str = None,
        figsize: tuple = (15, 6),
    ):
        """Plot time series with highlighted anomalies."""
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, height_ratios=[3, 1])

        # Main time series plot
        ax1.plot(data.index, data.values, "b-", alpha=0.7, label="Normal")

        # Highlight anomalies
        anomaly_indices = data.index[result.anomalies]
        ax1.scatter(
            anomaly_indices,
            data.loc[anomaly_indices],
            color="red",
            s=50,
            label="Anomaly",
            zorder=5,
        )

        ax1.set_title(title or f"{result.method} Anomaly Detection")
        ax1.set_ylabel("Value")
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # Anomaly scores plot
        if result.scores is not None:
            ax2.bar(data.index, result.scores, color="gray", alpha=0.5)
            if result.threshold:
                ax2.axhline(
                    y=result.threshold,
                    color="red",
                    linestyle="--",
                    label=f"Threshold: {result.threshold:.2f}",
                )
            ax2.set_ylabel("Anomaly Score")
            ax2.set_xlabel("Time")
            ax2.grid(True, alpha=0.3)
            ax2.legend()

        plt.tight_layout()
        return fig

    @staticmethod
    def plot_comparison(data: pd.Series, results: dict[str, AnomalyResult]):
        """Compare anomaly detection results from multiple methods."""
        n_methods = len(results)
        fig = make_subplots(
            rows=n_methods,
            cols=1,
            subplot_titles=list(results.keys()),
            vertical_spacing=0.05,
        )

        for i, (method_name, result) in enumerate(results.items(), 1):
            # Normal points
            normal_mask = ~result.anomalies
            fig.add_trace(
                go.Scatter(
                    x=data.index[normal_mask],
                    y=data.values[normal_mask],
                    mode="markers",
                    marker=dict(size=3, color="blue"),
                    name="Normal",
                    showlegend=(i == 1),
                ),
                row=i,
                col=1,
            )

            # Anomalies
            anomaly_mask = result.anomalies
            fig.add_trace(
                go.Scatter(
                    x=data.index[anomaly_mask],
                    y=data.values[anomaly_mask],
                    mode="markers",
                    marker=dict(size=8, color="red"),
                    name="Anomaly",
                    showlegend=(i == 1),
                ),
                row=i,
                col=1,
            )

        fig.update_layout(
            height=200 * n_methods,
            title="Anomaly Detection Method Comparison",
            showlegend=True,
        )

        return fig

    @staticmethod
    def plot_anomaly_heatmap(report: pd.DataFrame):
        """Create heatmap of anomaly detection results."""
        anomaly_cols = [col for col in report.columns if col.endswith("_anomaly")]

        # Sample if too many points
        if len(report) > 1000:
            sample_idx = np.random.choice(len(report), 1000, replace=False)
            sample_idx.sort()
            report_sample = report.iloc[sample_idx]
        else:
            report_sample = report

        # Create heatmap
        fig, ax = plt.subplots(figsize=(15, 8))

        heatmap_data = report_sample[anomaly_cols].T.astype(int)
        sns.heatmap(
            heatmap_data,
            cmap="RdYlGn_r",
            cbar_kws={"label": "Anomaly"},
            yticklabels=[col.replace("_anomaly", "") for col in anomaly_cols],
            xticklabels=False,
            ax=ax,
        )

        ax.set_title("Anomaly Detection Results Heatmap")
        ax.set_xlabel("Time Points")
        ax.set_ylabel("Detection Method")

        plt.tight_layout()
        return fig

## 3. Real-Time Anomaly Detection

In [ ]:
class RealTimeAnomalyDetector:
    """Real-time anomaly detection with sliding window."""

    def __init__(
        self,
        window_size: int = 100,
        update_frequency: int = 10,
        methods: list[str] = None,
    ):
        """Initialize real-time detector.

        Parameters:
        -----------
        window_size : int
            Size of sliding window for detection
        update_frequency : int
            How often to update the model
        methods : List[str]
            Methods to use for real-time detection
        """
        self.window_size = window_size
        self.update_frequency = update_frequency
        self.methods = methods or ["zscore", "isolation_forest"]

        self.buffer = deque(maxlen=window_size)
        self.anomaly_history = deque(maxlen=1000)
        self.detector = AnomalyDetector(contamination=0.05)
        self.update_counter = 0

    def process_point(self, value: float, timestamp: pd.Timestamp = None) -> dict:
        """Process a single data point.

        Parameters:
        -----------
        value : float
            New data point
        timestamp : pd.Timestamp
            Timestamp of the data point

        Returns:
        --------
        dict : Detection results
        """
        if timestamp is None:
            timestamp = pd.Timestamp.now()

        # Add to buffer
        self.buffer.append(value)

        # Need minimum points for detection
        if len(self.buffer) < 10:
            return {
                "timestamp": timestamp,
                "value": value,
                "is_anomaly": False,
                "confidence": 0.0,
            }

        # Create series from buffer
        data = pd.Series(list(self.buffer))

        # Run detection methods
        anomaly_votes = []

        for method in self.methods:
            try:
                if method == "zscore":
                    result = self.detector.zscore_detection(data, threshold=2.5)
                elif method == "isolation_forest":
                    result = self.detector.isolation_forest(data, n_estimators=50)
                elif method == "lof":
                    result = self.detector.local_outlier_factor(data, n_neighbors=5)
                else:
                    continue

                # Check if last point is anomaly
                anomaly_votes.append(result.anomalies[-1])
            except:
                pass

        # Determine if anomaly (majority voting)
        is_anomaly = sum(anomaly_votes) > len(anomaly_votes) / 2
        confidence = sum(anomaly_votes) / len(anomaly_votes) if anomaly_votes else 0.0

        # Store result
        result = {
            "timestamp": timestamp,
            "value": value,
            "is_anomaly": is_anomaly,
            "confidence": confidence,
            "methods_agree": anomaly_votes,
        }

        self.anomaly_history.append(result)

        # Update models periodically
        self.update_counter += 1
        if self.update_counter >= self.update_frequency:
            self._update_models()
            self.update_counter = 0

        return result

    def _update_models(self):
        """Update detection models with recent data."""
        # Re-initialize detector with recent statistics
        if len(self.buffer) > 0:
            recent_anomaly_rate = sum(
                1 for r in self.anomaly_history if r["is_anomaly"]
            ) / len(self.anomaly_history)
            self.detector.contamination = min(0.2, max(0.01, recent_anomaly_rate))

    def get_statistics(self) -> dict:
        """Get current detection statistics."""
        if not self.anomaly_history:
            return {}

        total = len(self.anomaly_history)
        anomalies = sum(1 for r in self.anomaly_history if r["is_anomaly"])

        return {
            "total_points": total,
            "anomaly_count": anomalies,
            "anomaly_rate": (anomalies / total) * 100,
            "buffer_size": len(self.buffer),
            "methods": self.methods,
        }

## 4. Evaluation Metrics

In [ ]:
class AnomalyEvaluator:
    """Evaluation metrics for anomaly detection."""

    @staticmethod
    def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
        """Calculate evaluation metrics.

        Parameters:
        -----------
        y_true : np.ndarray
            True anomaly labels (0 or 1)
        y_pred : np.ndarray
            Predicted anomaly labels (0 or 1)

        Returns:
        --------
        dict : Evaluation metrics
        """
        from sklearn.metrics import (
            accuracy_score,
            confusion_matrix,
            f1_score,
            precision_score,
            recall_score,
        )

        # Basic metrics
        metrics = {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1_score": f1_score(y_true, y_pred, zero_division=0),
        }

        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        metrics.update(
            {
                "true_positives": tp,
                "true_negatives": tn,
                "false_positives": fp,
                "false_negatives": fn,
            }
        )

        # Additional metrics
        metrics["specificity"] = tn / (tn + fp) if (tn + fp) > 0 else 0
        metrics["false_positive_rate"] = fp / (fp + tn) if (fp + tn) > 0 else 0
        metrics["false_negative_rate"] = fn / (fn + tp) if (fn + tp) > 0 else 0

        return metrics

    @staticmethod
    def cross_validate(
        data: pd.Series, detector: AnomalyDetector, n_splits: int = 5
    ) -> pd.DataFrame:
        """Cross-validation for anomaly detection.

        Parameters:
        -----------
        data : pd.Series
            Time series data
        detector : AnomalyDetector
            Detector instance
        n_splits : int
            Number of CV splits

        Returns:
        --------
        pd.DataFrame : CV results
        """
        from sklearn.model_selection import TimeSeriesSplit

        tscv = TimeSeriesSplit(n_splits=n_splits)
        cv_results = []

        for train_idx, test_idx in tscv.split(data):
            train_data = data.iloc[train_idx]
            test_data = data.iloc[test_idx]

            # Detect anomalies on test set
            result = detector.isolation_forest(test_data)

            # Store results
            cv_results.append(
                {
                    "train_size": len(train_idx),
                    "test_size": len(test_idx),
                    "anomaly_rate": result.anomaly_rate,
                    "anomaly_count": np.sum(result.anomalies),
                }
            )

        return pd.DataFrame(cv_results)

## 5. Example Usage

In [ ]:
# Generate synthetic time series with anomalies
np.random.seed(42)
n_points = 2000
time_index = pd.date_range("2023-01-01", periods=n_points, freq="H")

# Create base signal
t = np.arange(n_points)
trend = 0.01 * t
seasonal = 10 * np.sin(2 * np.pi * t / 24) + 5 * np.sin(2 * np.pi * t / 168)
noise = np.random.normal(0, 2, n_points)
signal = 100 + trend + seasonal + noise

# Insert anomalies
anomaly_indices = np.random.choice(n_points, size=int(n_points * 0.05), replace=False)
for idx in anomaly_indices:
    if np.random.random() > 0.5:
        signal[idx] += np.random.uniform(20, 40)  # Point anomaly
    else:
        signal[idx] *= np.random.uniform(0.3, 0.5)  # Point anomaly

# Create series
ts_data = pd.Series(signal, index=time_index, name="value")

print(f"Generated time series with {len(ts_data)} points")
print(
    f"Injected {len(anomaly_indices)} anomalies ({len(anomaly_indices) / n_points * 100:.1f}%)"
)
print("\nBasic statistics:")
print(ts_data.describe())

In [ ]:
# Initialize detector and run comprehensive detection
detector = AnomalyDetector(contamination=0.05)
report = detector.detect_all(ts_data, include_deep_learning=False)

print(f"\nDetection Report Shape: {report.shape}")
print("\nAnomalies detected by each method:")
for col in report.columns:
    if col.endswith("_anomaly"):
        method = col.replace("_anomaly", "")
        count = report[col].sum()
        pct = (count / len(report)) * 100
        print(f"  {method:20s}: {count:4d} anomalies ({pct:.1f}%)")

# Final consensus
print(
    f"\nConsensus anomalies: {report['is_anomaly'].sum()} ({report['is_anomaly'].mean() * 100:.1f}%)"
)

In [ ]:
# Visualize results
visualizer = AnomalyVisualizer()

# Plot individual method results
fig = visualizer.plot_anomalies(
    ts_data, detector.results["ensemble"], title="Ensemble Anomaly Detection Results"
)
plt.show()

# Plot comparison
selected_results = {
    "Z-Score": detector.results["zscore"],
    "Isolation Forest": detector.results["isolation_forest"],
    "Matrix Profile": detector.results.get("matrix_profile"),
    "Ensemble": detector.results["ensemble"],
}
selected_results = {k: v for k, v in selected_results.items() if v is not None}

fig = visualizer.plot_comparison(ts_data, selected_results)
fig.show()

In [ ]:
# Create anomaly heatmap
fig = visualizer.plot_anomaly_heatmap(report.iloc[:500])  # Show first 500 points
plt.show()

## 6. Real-Time Detection Example

In [ ]:
# Simulate real-time detection
rt_detector = RealTimeAnomalyDetector(
    window_size=50, update_frequency=10, methods=["zscore", "isolation_forest"]
)

# Process streaming data
print("Simulating real-time anomaly detection...\n")
detection_results = []

for i in range(200):  # Process first 200 points
    result = rt_detector.process_point(
        value=ts_data.iloc[i], timestamp=ts_data.index[i]
    )
    detection_results.append(result)

    # Print anomalies as they're detected
    if result["is_anomaly"] and i >= 50:  # After warmup
        print(
            f"🚨 Anomaly detected at {result['timestamp']}: "
            f"value={result['value']:.2f}, confidence={result['confidence']:.1%}"
        )

# Get statistics
stats = rt_detector.get_statistics()
print("\nReal-time Detection Statistics:")
print(f"  Total points processed: {stats['total_points']}")
print(f"  Anomalies detected: {stats['anomaly_count']}")
print(f"  Anomaly rate: {stats['anomaly_rate']:.1f}%")

## 7. Performance Evaluation

In [ ]:
# Create ground truth labels
true_anomalies = np.zeros(len(ts_data), dtype=bool)
true_anomalies[anomaly_indices] = True

# Evaluate each method
evaluator = AnomalyEvaluator()
evaluation_results = {}

print("Performance Evaluation Against Ground Truth:\n")
print(
    f"{'Method':<20} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Accuracy':<10}"
)
print("=" * 60)

for method_name, result in detector.results.items():
    if result and len(result.anomalies) == len(true_anomalies):
        metrics = evaluator.evaluate(true_anomalies, result.anomalies)
        evaluation_results[method_name] = metrics

        print(
            f"{method_name:<20} {metrics['precision']:<10.3f} "
            f"{metrics['recall']:<10.3f} {metrics['f1_score']:<10.3f} "
            f"{metrics['accuracy']:<10.3f}"
        )

# Find best method
if evaluation_results:
    best_method = max(
        evaluation_results, key=lambda x: evaluation_results[x]["f1_score"]
    )
    print(
        f"\nBest method by F1-Score: {best_method} "
        f"(F1={evaluation_results[best_method]['f1_score']:.3f})"
    )

## 8. Summary

This comprehensive anomaly detection system provides:

### Methods Implemented:
1. **Statistical Methods**:
   - Z-Score detection
   - IQR-based detection
   - Grubbs test

2. **Machine Learning Methods**:
   - Isolation Forest
   - Local Outlier Factor (LOF)
   - Elliptic Envelope
   - One-Class SVM

3. **Time Series Specific**:
   - Seasonal Hybrid ESD (S-H-ESD)
   - Matrix Profile
   - LSTM Autoencoder

4. **Ensemble Methods**:
   - Majority voting
   - Weighted voting
   - Unanimous voting

### Key Features:
- Comprehensive evaluation metrics
- Real-time detection capabilities
- Interactive visualizations
- Cross-validation support
- Automatic parameter tuning

### Use Cases:
- Network intrusion detection
- Equipment failure prediction
- Financial fraud detection
- Quality control monitoring
- System performance anomalies

The system is production-ready and can be easily integrated into existing pipelines!